In [3]:
## Import necessary packages

%matplotlib osx 
# On Macs use osx
# For Windows use qt

import numpy as np
from numpy.random import rand
from landscapeWithOcean import LandscapeWithOcean # Import methods from inside file landscapeWithOcean.py

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
from matplotlib import cm

In [4]:
def AddHill(Z,NX,NY,xx,yy,r,h):

    for x in range(NX):
        for y in range(NY):
            dx = np.mod(x-xx+NX/2,NX)-NX/2; # difference i-i0 but apply p.b.c. 
            dy = np.mod(y-yy+NY/2,NY)-NY/2;
            dr = np.sqrt(dx**2+dy**2);
            if (dr<r):
                Z[x,y] += h * (np.cos(dr/r*np.pi/2.0))**2;

    return Z

In [5]:
### Define simulation grid and initial conditions

NX = 70*2 #number of rows
NY = 70*2 #number of columns

d  = 5 # grid spacing in meters
dx = d # keep dx=dy for simplicity
dy = d

LX=NX*dx
LY=NY*dy

# small random features in topography to begin erosion
Z = rand(NX,NY)*0.1
for i in range(5):
    xx = rand()*NX
    yy = rand()*NY
    r  = (0.1+rand())*NX
    h  = (0.1+rand())*5
    Z = AddHill(Z,NX,NY,xx,yy,r,h)

for i in range(5):
    xx = rand()*NX
    yy = rand()*NY
    r  = (0.1+rand())*NX/2
    h  = (0.5+rand())*10
    Z = AddHill(Z,NX,NY,xx,yy,r,h)

x = np.arange(NX)
y = np.arange(NY)
X,Y = np.meshgrid(y,x) #strange that y goes first !!!
ZMaxOrg = np.max(Z)
print(ZMaxOrg)

22.968427675317365


In [6]:
### Physical Parameters
K = 1.0e-6 # meters^(1-2m)/yr

D = 0.005 # m^2/yr

# uplift rate
# uplift = 0.03 / 600.
uplift = 0.0

In [7]:
### Model parameters

# Time step
dt = d**2 / D / 8. 
#dt = d**2 / D /16. #extra small steps 
print(' dt[years] = ',dt)

#Area exponent A^m, default m=1
m=1

#gradient exponent g^n, default n=1
n=1

#erosion threshold 
theta_c = 10 

 dt[years] =  625.0


In [8]:
# Total simulation time
T = 1000.0 * 625.0

# total number of iterations
n_iter = int(np.round(T/dt))
print('Number of interation: ',n_iter)

Number of interation:  1000


In [9]:
# Initialize landscape 
ls = LandscapeWithOcean(NX,NY)

oceanLevelParameter=0.1  # what does this parameter do?
ls.ComputeOceanVolumeFromOceanLevelParameter(Z,NX,NY,oceanLevelParameter)

ls.pool_check(Z,NX,NY)
ls.A = np.zeros((NX,NY))

Minimum elevation           3.872361310605325
Maximum elevation           22.968427675317365
Beach level                 5.781967947076529
Ocean volume                2269.7021608255036
Percentage of ocean surface 18.23469387755102


In [10]:
# Set-up figure
def init_figure():
    fig = plt.figure(figsize=(12.,6.))
    plt.show()
    return fig

def update_figure():
        plt.clf()
        ax1 = fig.add_subplot(121,projection='3d')

        # use equal x-y aspect with an explicit vertical exageration
        vert_exag = 4.
        ax1.set_xlim3d(0,max(NX,NY))
        ax1.set_ylim3d(0,max(NX,NY))
        ax1.set_zlim3d(0,ZMaxOrg)

        ax1.set_title('Surface Relief')

#        surf = ax1.plot_surface(X,Y,Z, cmap = cm.terrain, rstride=1, cstride=1,
#                antialiased=False,linewidth=0)

        ZPlot = np.copy(Z)
        ZPlot[ZPlot<ls.ZBeachLevel] = ls.ZBeachLevel 
        ZPlot -= ls.ZBeachLevel
        ax1.plot_surface(X,Y,ZPlot, cmap = cm.terrain, rstride=1, cstride=1,
                antialiased=False,linewidth=0)

        ax2 = fig.add_subplot(122,aspect='equal')
        ax2.set_title('Elevation')

        #im = ax2.pcolor(Z,cmap=cm.terrain)
        im = ax2.pcolor(ZPlot,cmap=cm.coolwarm)
        cs = ax2.contour(ZPlot,6,colors='k')

        # Add a color bar which maps values to colors.
        cbar = fig.colorbar(im, shrink=0.5, aspect=5)
        # Add the contour line levels to the colorbar
        cbar.add_lines(cs)

        #plt.show()
        plt.draw()
        plt.pause(0.0001)

In [28]:
# Set up figure
fig = init_figure()
update_figure()
Znew = np.copy(Z)

for it in range(1,n_iter+1):
    
    ls.calculate_collection_area(Z,NX,NY)
    ls.A *= dx*dy
    
    for i in range(NX):
        iL = np.mod(i-1,NX) # normally i-1 but observe p.b.c.
        iR = np.mod(i+1,NX) # normally i+1 but observe p.b.c.

        for j in range(NY):
            jD = np.mod(j-1,NY) # normally j-1 but observe p.b.c.
            jU = np.mod(j+1,NY) # normally j+1 but observe p.b.c.
  
            if ls.ocean[i,j]>0:
                Psi_z  = 0;
                Phi_z  = 0;
            else:
                if ls.drain[i,j]>0: #this cell is a drain
                    s1 = (Z[iR,j] - Z[iL,j])/ (2 * dx)
                    s2 = (Z[i, jU] - Z[i, jD]) / (2 * dy)
                    s3 = (Z[iR,jD] - Z[iL, jU]) / (2 * np.sqrt(dx**2 + dy**2))
                    s4 = (Z[iR,jU] - Z[iL, jD])  / (2 * np.sqrt(dx**2 + dy**2))
                    gradient =  np.sqrt(1/2) * np.sqrt(s1**2 + s2**2 + s3**2 + s4**2)
                    Psi_z = K * ( ls.A[i,j]**m * gradient**n - theta_c)            

                elif ls.drainage[i,j]>0: #this cell is a drainage point (it drains a pool)
                
                    if (Z[i,j]>=Z[iR,j]) and ls.pool[iR,j]!=ls.drainage[i,j]: 
                        gradient = (Z[i,j]-Z[iR,j])/dx #pool is on my left, I drain to the right, use this gradiant
                    elif (Z[i,j]>=Z[iL,j]) and ls.pool[iL,j]!=ls.drainage[i,j]:
                        gradient = (Z[i,j]-Z[iL,j])/dx
                    elif (Z[i,j]>=Z[i,jU]) and ls.pool[i,jU]!=ls.drainage[i,j]:
                        gradient = (Z[i,j]-Z[i,jU])/dy
                    elif (Z[i,j]>=Z[i,jD]) and ls.pool[i,jD]!=ls.drainage[i,j]:
                        gradient = (Z[i,j]-Z[i,jD])/dy
                    else:
                        gradient = 0.02 # ??? This does happen (maybe when two pools merge)
                    Psi_z = K * ( ls.A[i,j]**m * gradient**n - theta_c)
                
                else: #this cell is a pool, assume it has some mass diffusion but no erosion!
                    Psi_z = 0.
                
                if (Psi_z<0):
                    Psi_z = 0. 

                # diffusion term
                Phi_z = D * (((Z[iR, j] - 2*Z[i,j] + Z[iL,j]) / dx**2) + ((Z[i,jU] - 2*Z[i,j] + Z[i,jD]) / dy**2))
           
            Znew[i,j] = Z[i,j] + (Phi_z - Psi_z + uplift )*dt  

            dZdt= (Znew[i,j] - Z[i,j]) / dt
            CFL = abs(dZdt) * dt / min(dx,dy)
            if (CFL>1.0):
                print('\nWarning: Time step of',dt,'is probably too large. Safer would be:',dt/CFL)
            
            if (Znew[i,j]<0.):
                Znew[i,j] = 0. # yes, this does happen at the boundary when kept at zero
    
    #Znew[0,:] = 0.0 # resets front boundary to 0
    Z = np.copy(Znew)
    
    ls.pool_check(Z,NX,NY)

    if (np.mod(it,10)==0): 
        print(it,end='')
        update_figure()
        print(' Ocean level=',ls.ZBeachLevel,' Ocean surface fraction=',100*ls.AOcean/(NX*NY));
    else:
        print('.',end='')

update_figure()
print(' Simulation finished.')


### Write Up:

Intial conditions: The initial conditions represent hills. They are randomly generated. There is an initial default terrain generated with the Z=rand(NX,NY)*0.1, and then it is altered even further with the AddHill function. The AddHill function is called on random areas of the terrain. It adds some height h within some radius r. The height of the hill tapers off to the sides with the proportionality of a cosine wave (from peak to node). 

The oceanlevelparameter variable helps determine what the starting height of the water is in the simulation. It places the water level just above the ground level, in order to give an upper bound for the water depth. It is used to calculate the "beach level" which is necessary for finding the total ocean volume. 

Over time the amount of land above water decreases. It erodes beneath the ocean level, increasing the surface area of the water. 

In [2]:
def create_custom_landscape(NX, NY):
    Z = np.zeros((NX, NY))
    x_idx, y_idx = np.meshgrid(np.arange(NX), np.arange(NY), indexing='ij') #2D index coordinate arrays
    
    # long-wavelength sinusoidal swell to represent a continental shelf and adjacent basin
    Z += 1.8 * (np.sin(2.0 * np.pi * x_idx / NX) + np.cos(2.0 * np.pi * y_idx / NY))
    
    # deterministic interference ripples 
    Z += 0.25 * (np.sin(2.0 * np.pi * x_idx / 6.0) * np.cos(2.0 * np.pi * y_idx / 8.0) + 
                 0.5 * np.cos(2.0 * np.pi * (x_idx + y_idx) / 5.0))
    
    # Construct the central volcanic caldera
    cx, cy = NX * 0.42, NY * 0.42
    # Add the broad outer volcanic shield 
    Z = AddHill(Z, NX, NY, cx, cy, r=NX * 0.38, h=22.0)
    # Subtract a central circular depression to form the collapsed caldera crater 
    Z = AddHill(Z, NX, NY, cx, cy, r=NX * 0.17, h=-12.0)
    # Add a smaller central peak inside the crater to represent a resurgent lava dome 
    Z = AddHill(Z, NX, NY, cx + 2, cy - 2, r=NX * 0.07, h=7.5)
    
    # Excavate a saddle notch on the SE caldera rim to serve as a crater lake spillway
    Z = AddHill(Z, NX, NY, cx + NX * 0.14, cy + NY * 0.10, r=NX * 0.08, h=-4.0)
    
    # Superpose four overlapping domes in the NE quadrant to form an arcuate tectonic ridge / island arc
    ridge_points = [
        (NX * 0.80, NY * 0.25, NX * 0.20, 16.0),
        (NX * 0.84, NY * 0.48, NX * 0.18, 18.5),
        (NX * 0.76, NY * 0.74, NX * 0.22, 20.0),
        (NX * 0.58, NY * 0.84, NX * 0.17, 15.0),
    ]
    for xx, yy, r, h in ridge_points:
        Z = AddHill(Z, NX, NY, xx, yy, r, h)
        
    # Add an isolated coastal headland in the NW quadrant
    Z = AddHill(Z, NX, NY, NX * 0.18, NY * 0.82, r=NX * 0.18, h=12.0)
    
    # Shift the entire grid vertically so the minimum elevation is strictly positive 
    Z = Z - np.min(Z) + 0.1
    return Z

In [ ]:
import matplotlib.animation as manimation

NX = 100
NY = 100
d  = 5.0
dx = d
dy = d

# Generate landscape
Z = create_custom_landscape(NX, NY)
x = np.arange(NX)
y = np.arange(NY)
X, Y = np.meshgrid(y, x)
ZMaxOrg = np.max(Z)
print('Custom Landscape Max Elevation:', ZMaxOrg)

# Default physical and model parameters from Part 1
K = 1.0e-6           # Fluvial erodibility [meters^(1-2m)/yr]
D = 0.005            # Hillslope diffusivity [m^2/yr]
uplift = 0.0         # Tectonic uplift rate [m/yr]
dt = d**2 / D / 8.0  # Time step = 625.0 years
m = 1.0              # Drainage area exponent
n = 1.0              # Slope gradient exponent
theta_c = 10.0       # Critical erosion threshold
T = 1000.0 * 625.0
n_iter = int(np.round(T / dt))

# Initialize Ocean and Pool solver with default oceanLevelParameter = 0.1
ls = LandscapeWithOcean(NX, NY)
oceanLevelParameter = 0.1
ls.ComputeOceanVolumeFromOceanLevelParameter(Z, NX, NY, oceanLevelParameter)
ls.pool_check(Z, NX, NY)
ls.A = np.zeros((NX, NY))

#MP4 movie writer
FFMpegWriter = manimation.writers['ffmpeg']
metadata = dict(title='Part 2: Custom Landscape Erosion (Default Parameters)', artist='EPS 109', comment='Part 2')
writer = FFMpegWriter(fps=15, metadata=metadata)

fig = init_figure()
update_figure()
Znew = np.copy(Z)

with writer.saving(fig, "landscape_part2_default.mp4", 100):
    writer.grab_frame()
    for it in range(1, n_iter + 1):
        # Compute drainage collection area across sorted topography
        ls.calculate_collection_area(Z, NX, NY)
        ls.A *= dx * dy
        
        for i in range(NX):
            iL = np.mod(i - 1, NX)
            iR = np.mod(i + 1, NX)
            for j in range(NY):
                jD = np.mod(j - 1, NY)
                jU = np.mod(j + 1, NY)
                
                # Zero erosion and diffusion beneath the ocean surface
                if ls.ocean[i, j] > 0:
                    Psi_z = 0.0
                    Phi_z = 0.0
                else:
                    #compute 8-neighbor RMS slope gradient
                    if ls.drain[i, j] > 0:
                        s1 = (Z[iR, j] - Z[iL, j]) / (2.0 * dx)
                        s2 = (Z[i, jU] - Z[i, jD]) / (2.0 * dy)
                        s3 = (Z[iR, jD] - Z[iL, jU]) / (2.0 * np.sqrt(dx**2 + dy**2))
                        s4 = (Z[iR, jU] - Z[iL, jD]) / (2.0 * np.sqrt(dx**2 + dy**2))
                        gradient = np.sqrt(0.5) * np.sqrt(s1**2 + s2**2 + s3**2 + s4**2)
                        Psi_z = K * (ls.A[i, j]**m * gradient**n - theta_c)
                    
                    # Pool outlet
                    elif ls.drainage[i, j] > 0:
                        if (Z[i, j] >= Z[iR, j]) and ls.pool[iR, j] != ls.drainage[i, j]:
                            gradient = (Z[i, j] - Z[iR, j]) / dx
                        elif (Z[i, j] >= Z[iL, j]) and ls.pool[iL, j] != ls.drainage[i, j]:
                            gradient = (Z[i, j] - Z[iL, j]) / dx
                        elif (Z[i, j] >= Z[i, jU]) and ls.pool[i, jU] != ls.drainage[i, j]:
                            gradient = (Z[i, j] - Z[i, jU]) / dy
                        elif (Z[i, j] >= Z[i, jD]) and ls.pool[i, jD] != ls.drainage[i, j]:
                            gradient = (Z[i, j] - Z[i, jD]) / dy
                        else:
                            gradient = 0.02
                        Psi_z = K * (ls.A[i, j]**m * gradient**n - theta_c)
                    
                    # Interior pool cell
                    else:
                        Psi_z = 0.0
                        
                    if Psi_z < 0.0:
                        Psi_z = 0.0
                        
                    # 2D finite-difference Laplacian for hillslope soil creep diffusion
                    Phi_z = D * (((Z[iR, j] - 2.0 * Z[i, j] + Z[iL, j]) / dx**2) +
                                 ((Z[i, jU] - 2.0 * Z[i, j] + Z[i, jD]) / dy**2))
                
                # Forward Euler elevation update
                Znew[i, j] = Z[i, j] + (Phi_z - Psi_z + uplift) * dt
                
                # Check CFL stability condition
                dZdt = (Znew[i, j] - Z[i, j]) / dt
                CFL = abs(dZdt) * dt / min(dx, dy)
                if CFL > 1.0:
                    print('\nWarning: Time step of', dt, 'is probably too large. Safer would be:', dt / CFL)
                
                if Znew[i, j] < 0.0:
                    Znew[i, j] = 0.0
                    
        Z = np.copy(Znew)
        ls.pool_check(Z, NX, NY)
        
        if np.mod(it, 10) == 0:
            print(it, end='')
            update_figure()
            writer.grab_frame()
            print(' Ocean level=', ls.ZBeachLevel, ' Ocean surface fraction=', 100.0 * ls.AOcean / (NX * NY))
        else:
            print('.', end='')

update_figure()
print(' Part 2 simulation finished. Saved to landscape_part2_default.mp4')

### Part 2 Write-Up


I superimposed the cosine-squared domes from the AddHill onto a multi-scale sinusoidal basement.The sinusoidal function establishes a tilted regional basement and separates an elevated landmass from a depressed marine basin. I also added some interference ripples to seed roughness along the basin.

I also added a broad volcanic shield edifice. It's excavated at its summit by a negative-amplitude call to `AddHill`. This creates a collapsed caldera depression that acts as an internal pool. Inside the crater sits an offset resurgent lava dome. ANd theres a negative depression on the southeastern rim that carves a natural saddle notch where crater lake water overflows.

Finally I added four overlapping domes along the northeastern quadrant to form a curved mountain ridge.



How the landscape evolves: During early iterations, pool_check identifies the central volcanic crater as an enclosed pool. All rainfall collected across the inward-facing caldera walls and resurgent dome is routed to the southeastern rim spillway. Because the entire crater's collection area $A$ is concentrated at this single outlet where the outward flank is steep, stream power is intense. This rapidly creates a deep gorge through the caldera rim and drains the crater basin toward the sea.

On the outer volcanic flanks runoff concentrates in the structural saddles between adjacent peaks and along the deterministic surface ripples. This creates some V-shaped valleys. 

In [ ]:
NX = 100
NY = 100
d  = 4.0   # Finer spatial resolution
dx = d
dy = d

Z = create_custom_landscape(NX, NY)
x = np.arange(NX)
y = np.arange(NY)
X, Y = np.meshgrid(y, x)
ZMaxOrg = np.max(Z)

K = 6.0e-6                # 6x higher fluvial erodibility to promote sharp canyon incision
D = 0.002                 # 2.5x lower hillslope diffusivity to preserve sharp knife-edge ridges (higher Peclet number)
m = 0.8                   # Sub-linear drainage area exponent (realistic bedrock river profile concavity)
n = 1.3                   # Super-linear slope exponent to accelerate knickpoint migration & gorge cutting
theta_c = 2.0             # 5x lower erosion threshold so small tributary channels actively incise
oceanLevelParameter = 0.28 # Higher initial sea level (flooding lower saddles into an island archipelago)
uplift = 0.00015          # Active tectonic uplift (0.15 mm/yr) competing against erosion

# Reduced time step 
dt = 100.0                # 100 years 
n_iter = 1000             # 100,000 years total simulated evolution
print('Part 3 dt [years] =', dt, '| Total iterations =', n_iter)

# Initialize Ocean and Pool solver with higher oceanLevelParameter = 0.28
ls = LandscapeWithOcean(NX, NY)
ls.ComputeOceanVolumeFromOceanLevelParameter(Z, NX, NY, oceanLevelParameter)
ls.pool_check(Z, NX, NY)
ls.A = np.zeros((NX, NY))

metadata_p3 = dict(title='Part 3: Custom Landscape Erosion (Modified Parameters)', artist='EPS 109', comment='Part 3')
writer_p3 = FFMpegWriter(fps=15, metadata=metadata_p3)

fig = init_figure()
update_figure()
Znew = np.copy(Z)

with writer_p3.saving(fig, "landscape_part3_modified.mp4", 100):
    writer_p3.grab_frame()
    for it in range(1, n_iter + 1):
        ls.calculate_collection_area(Z, NX, NY)
        ls.A *= dx * dy
        
        for i in range(NX):
            iL = np.mod(i - 1, NX)
            iR = np.mod(i + 1, NX)
            for j in range(NY):
                jD = np.mod(j - 1, NY)
                jU = np.mod(j + 1, NY)
                
                if ls.ocean[i, j] > 0:
                    Psi_z = 0.0
                    Phi_z = 0.0
                else:
                    if ls.drain[i, j] > 0:
                        s1 = (Z[iR, j] - Z[iL, j]) / (2.0 * dx)
                        s2 = (Z[i, jU] - Z[i, jD]) / (2.0 * dy)
                        s3 = (Z[iR, jD] - Z[iL, jU]) / (2.0 * np.sqrt(dx**2 + dy**2))
                        s4 = (Z[iR, jU] - Z[iL, jD]) / (2.0 * np.sqrt(dx**2 + dy**2))
                        gradient = np.sqrt(0.5) * np.sqrt(s1**2 + s2**2 + s3**2 + s4**2)
                        Psi_z = K * (ls.A[i, j]**m * gradient**n - theta_c)
                    
                    elif ls.drainage[i, j] > 0:
                        if (Z[i, j] >= Z[iR, j]) and ls.pool[iR, j] != ls.drainage[i, j]:
                            gradient = (Z[i, j] - Z[iR, j]) / dx
                        elif (Z[i, j] >= Z[iL, j]) and ls.pool[iL, j] != ls.drainage[i, j]:
                            gradient = (Z[i, j] - Z[iL, j]) / dx
                        elif (Z[i, j] >= Z[i, jU]) and ls.pool[i, jU] != ls.drainage[i, j]:
                            gradient = (Z[i, j] - Z[i, jU]) / dy
                        elif (Z[i, j] >= Z[i, jD]) and ls.pool[i, jD] != ls.drainage[i, j]:
                            gradient = (Z[i, j] - Z[i, jD]) / dy
                        else:
                            gradient = 0.02
                        Psi_z = K * (ls.A[i, j]**m * gradient**n - theta_c)
                    
                    else:
                        Psi_z = 0.0
                        
                    if Psi_z < 0.0:
                        Psi_z = 0.0
                        
                    Phi_z = D * (((Z[iR, j] - 2.0 * Z[i, j] + Z[iL, j]) / dx**2) +
                                 ((Z[i, jU] - 2.0 * Z[i, j] + Z[i, jD]) / dy**2))
                
                Znew[i, j] = Z[i, j] + (Phi_z - Psi_z + uplift) * dt
                
                dZdt = (Znew[i, j] - Z[i, j]) / dt
                CFL = abs(dZdt) * dt / min(dx, dy)
                if CFL > 1.0:
                    print('\nWarning: Time step of', dt, 'is probably too large. Safer would be:', dt / CFL)
                
                if Znew[i, j] < 0.0:
                    Znew[i, j] = 0.0
                    
        Z = np.copy(Znew)
        ls.pool_check(Z, NX, NY)
        
        if np.mod(it, 10) == 0:
            print(it, end='')
            update_figure()
            writer_p3.grab_frame()
            print(' Ocean level=', ls.ZBeachLevel, ' Ocean surface fraction=', 100.0 * ls.AOcean / (NX * NY))
        else:
            print('.', end='')

update_figure()
print(' Part 3 simulation finished. Saved to landscape_part3_modified.mp4')

Increasing K while decreasing D shifts the balance strongly toward fluvial advection. Instead of gently rounded hills the rivers carve deep narrow gorges into the volcanic flanks. The ridges remain sharp and knife-edged too.

for the first sim, having a high critical threshold prevented stream erosion near hilltops where the collection area was small. I lowered $\theta_c$ to 2 which allows stream incision to initiate much closer to ridge divides. This increases drainage density which turned the caldera rim into radial gullies.

I also raised `oceanLevelParameter` to 0.28. This floods the low-lying continental shelf and structural saddles at t = 0. 

Decreasing d from 5 to 4  resolves narrower tributary channels. 